# Collaborative filtering / matrix factorization

_Classical ML_

**Fill the blanks in a ratings table by factoring it into two thin matrices.**

You have a big grid: rows are users, columns are items, cells are ratings. Most cells are blank.

     Collaborative filtering fills the blanks by learning hidden tastes.

---

This notebook is a practice scaffold for the **Collaborative filtering / matrix factorization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — scikit-learn

### Step 1 — Build a low-rank ratings table with holes

Collaborative filtering assumes a ratings table is secretly low-rank: a few hidden tastes explain every rating. We manufacture exactly that — 40 users by 25 items, generated from rank-3 factors — then knock out 30% of the cells to mimic ratings nobody gave, zero-filling the unknowns.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# True ratings are exactly rank 3: 40 users x 3 tastes, 25 items x 3 tastes.
U = rng.normal(size=(40, 3))
V = rng.normal(size=(25, 3))
R = U @ V.T

# Reveal ~70% of the cells; the rest are 'unrated' and zero-filled.
mask = rng.random(R.shape) < 0.7
Robs = np.where(mask, R, 0.0)

### Step 2 — Factor the observed table with truncated SVD

Truncated SVD factors the observed table into thin user and item matrices using just 3 components — the same rank we built in. Multiplying the user factors `Z` back by the item factors (`svd.components_`) reconstructs a full table, filling in every blank cell.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=3, random_state=0)
Z = svd.fit_transform(Robs)            # user factors (40 x 3)
Rhat = Z @ svd.components_              # reconstructed full table

### Step 3 — Score the fill-in on seen vs hidden cells

The real test is the cells we hid. We compare the reconstruction against the true ratings separately on the observed cells (which SVD saw) and the hidden cells (which it did not). A low RMSE on the hidden cells means the factorisation genuinely recovered the missing tastes, not just memorised what it was shown.

In [ ]:
known = mask
hidden = ~mask

rmse_observed = np.sqrt(np.mean((Rhat[known] - R[known]) ** 2))
rmse_hidden = np.sqrt(np.mean((Rhat[hidden] - R[hidden]) ** 2))

print("rank used        :", svd.n_components)
print("RMSE on observed :", round(rmse_observed, 3))
print("RMSE on hidden   :", round(rmse_hidden, 3))

## Visualize the data & results

_Can we fill in the movie ratings these viewers never gave?_

### Step 1 — Set up a small movie-ratings table

Here is a hand-built, real-style example: 6 viewers, 8 movies, ratings from 1 to 5. The companion `mask` marks which cells each viewer actually rated; the rest are the blanks we want to fill in.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

users = ["Alice", "Ben", "Chloe", "Diego", "Emma", "Frank"]
movies = ["Matrix", "Titanic", "Inception", "ToyStory",
          "Gladiator", "Frozen", "Avatar", "Shrek"]

# Real-style 1-5 ratings, one row per viewer.
R = np.array([
    [5, 2, 5, 3, 4, 2, 4, 3],
    [4, 2, 4, 4, 3, 2, 3, 3],
    [2, 5, 2, 4, 2, 5, 3, 5],
    [3, 3, 3, 4, 3, 3, 4, 4],
    [1, 4, 1, 3, 2, 5, 2, 5],
    [4, 3, 5, 3, 5, 2, 4, 2]], dtype=float)

# Which cells the viewer actually rated.
mask = np.array([
    [1, 1, 0, 1, 1, 1, 1, 0],
    [1, 0, 1, 1, 0, 1, 1, 1],
    [1, 1, 1, 0, 1, 1, 0, 1],
    [0, 1, 1, 1, 1, 0, 1, 1],
    [1, 0, 1, 1, 1, 1, 1, 0],
    [1, 1, 0, 1, 1, 1, 0, 1]], dtype=bool)

### Step 2 — Center, factor, and reconstruct

We subtract the mean rating so the factorisation models how each rating differs from average, then zero-fill the unrated cells. A rank-3 truncated SVD factors this centered table and reconstructs it; adding the mean back puts the predictions on the original 1-5 scale.

In [ ]:
mean = R[mask].mean()
Robs = np.where(mask, R - mean, 0.0)     # center on observed, zero-fill unknowns

svd = TruncatedSVD(n_components=3, random_state=0)
Rhat = svd.fit_transform(Robs) @ svd.components_ + mean

### Step 3 — Show the observed and reconstructed tables side by side

The left heatmap shows only the ratings viewers gave (blanks left out), and the right one shows the rank-3 reconstruction with every cell filled. Comparing them lets you see the predicted ratings the SVD invented for movies each viewer never watched.

In [ ]:
obs = np.where(mask, R, np.nan)          # blanks as NaN so they render empty

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(obs, cmap="coolwarm", vmin=1, vmax=5)
ax[0].set_title("Observed movie ratings (blanks hidden)")
ax[1].imshow(Rhat, cmap="coolwarm", vmin=1, vmax=5)
ax[1].set_title("Reconstructed ratings (rank-3 SVD)")

for a in ax:
    a.set_xticks(range(len(movies)))
    a.set_xticklabels(movies, rotation=90)
    a.set_yticks(range(len(users)))
    a.set_yticklabels(users)

plt.show()